# 08 — Evaluation

This notebook covers the three evaluation modules:
1. **Evaluator** (`src/evaluation/metrics.py`) — perplexity, sample generation, loss plots
2. **LLMJudge** (`src/evaluation/human_eval.py`) — LLM-as-judge scoring
3. **ErrorAnalyzer** (`src/evaluation/error_analysis.py`) — failure categorization

## 1. Evaluator — Perplexity Computation

Perplexity measures how well the model predicts a test set.
It is defined as `exp(average cross-entropy loss)`.

In [ ]:
import math
import os
import sys

import torch
import torch.nn as nn
from torch import Tensor

# Ensure project root is on path
sys.path.insert(0, os.path.abspath('..'))

from src.config import GPTConfig
from src.model.transformer import GPTModel

In [ ]:
class Evaluator:
    """Evaluation utilities for GPT models."""

    @torch.no_grad()
    def compute_perplexity(self, model: GPTModel, test_data: Tensor) -> float:
        """Compute perplexity as exp(average cross-entropy loss)."""
        model.eval()
        max_seq_len = model.config.max_seq_len
        if test_data.dim() != 1 or test_data.size(0) < 2:
            return float('inf')
        device = next(model.parameters()).device
        test_data = test_data.to(device)
        loss_fn = nn.CrossEntropyLoss()
        total_loss, num_chunks = 0.0, 0
        for start in range(0, test_data.size(0) - 1, max_seq_len):
            end = min(start + max_seq_len + 1, test_data.size(0))
            chunk = test_data[start:end]
            if chunk.size(0) < 2:
                break
            input_ids = chunk[:-1].unsqueeze(0)
            targets = chunk[1:]
            logits = model(input_ids).squeeze(0)
            loss = loss_fn(logits, targets)
            total_loss += loss.item()
            num_chunks += 1
        if num_chunks == 0:
            return float('inf')
        return math.exp(total_loss / num_chunks)

    @staticmethod
    def perplexity_from_loss(avg_loss: float) -> float:
        """Compute perplexity from average cross-entropy loss."""
        return math.exp(avg_loss)

### Demo: Perplexity on random data

We create a small model and compute perplexity on random token IDs.

In [ ]:
config = GPTConfig(vocab_size=100, d_model=64, num_heads=4, num_layers=2, max_seq_len=32)
model = GPTModel(config)
evaluator = Evaluator()

# Random test data
test_data = torch.randint(0, 100, (200,))
ppl = evaluator.compute_perplexity(model, test_data)
print(f'Perplexity on random data: {ppl:.2f}')

# Verify the math: perplexity_from_loss
loss_val = 3.5
print(f'exp({loss_val}) = {evaluator.perplexity_from_loss(loss_val):.4f}')

## 2. LLMJudge — LLM-as-Judge Evaluation

The `LLMJudge` sends generated texts to an external LLM API and
records fluency, coherence, and instruction-following scores (1-5).

In [ ]:
import json
import logging
import requests

logger = logging.getLogger(__name__)

class LLMJudge:
    DEFAULT_API_URL = 'https://api.openai.com/v1/chat/completions'
    DEFAULT_MODEL = 'gpt-4o-mini'
    SYSTEM_PROMPT = (
        'You are an expert text quality evaluator. Rate on fluency, '
        'coherence, instruction_following (1-5). Respond with JSON only.'
    )

    def __init__(self, api_url=None, model=None):
        self.api_url = api_url or self.DEFAULT_API_URL
        self.model = model or self.DEFAULT_MODEL

    def evaluate(self, generated_texts, api_key):
        results = []
        for text in generated_texts:
            try:
                resp = requests.post(self.api_url, headers={
                    'Authorization': f'Bearer {api_key}',
                    'Content-Type': 'application/json'
                }, json={
                    'model': self.model,
                    'messages': [
                        {'role': 'system', 'content': self.SYSTEM_PROMPT},
                        {'role': 'user', 'content': f'Evaluate:\n{text}'}
                    ], 'temperature': 0.0
                }, timeout=30)
                resp.raise_for_status()
                scores = json.loads(resp.json()['choices'][0]['message']['content'])
                results.append(scores)
            except Exception as e:
                results.append({'error': str(e)})
        return results

print('LLMJudge class defined (requires API key to run)')

## 3. ErrorAnalyzer — Failure Categorization

Detects repetition (repeated n-grams), incoherence, off-topic, and grammatical errors.

In [ ]:
import re
from collections import Counter

class ErrorAnalyzer:
    def __init__(self, ngram_size=3, repetition_threshold=3):
        self.ngram_size = max(ngram_size, 3)
        self.repetition_threshold = repetition_threshold

    def analyze(self, generated_texts):
        results = {
            'repetition': {'count': 0, 'examples': []},
            'incoherence': {'count': 0, 'examples': []},
            'off_topic': {'count': 0, 'examples': []},
            'grammatical': {'count': 0, 'examples': []},
            'total_texts': len(generated_texts),
        }
        for text in generated_texts:
            if self._detect_repetition(text):
                results['repetition']['count'] += 1
                results['repetition']['examples'].append(text[:200])
            if self._detect_incoherence(text):
                results['incoherence']['count'] += 1
            if self._detect_off_topic(text):
                results['off_topic']['count'] += 1
            if self._detect_grammatical(text):
                results['grammatical']['count'] += 1
        return results

    def _detect_repetition(self, text):
        words = text.split()
        if len(words) < self.ngram_size:
            return False
        max_n = min(len(words), self.ngram_size + 4)
        for n in range(self.ngram_size, max_n):
            ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
            counts = Counter(ngrams)
            if any(c > self.repetition_threshold for c in counts.values()):
                return True
        return False

    def _detect_incoherence(self, text):
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        if not sentences:
            return False
        avg_words = sum(len(s.split()) for s in sentences) / len(sentences)
        return len(sentences) > 3 and avg_words < 2

    def _detect_off_topic(self, text):
        if not text.strip():
            return True
        alpha = sum(1 for c in text if c.isalpha())
        return alpha / len(text.strip()) < 0.3 if text.strip() else True

    def _detect_grammatical(self, text):
        if re.search(r'  {3,}', text):
            return True
        return text.count('(') != text.count(')')

### Demo: Error Analysis

In [ ]:
analyzer = ErrorAnalyzer(ngram_size=3, repetition_threshold=3)

sample_texts = [
    'The cat sat on the mat. The cat sat on the mat. The cat sat on the mat. The cat sat on the mat. The cat sat on the mat.',
    'Hello world this is a normal sentence with good structure and flow.',
    '!!! ??? ... @@@ ### $$$',
    'This has (unbalanced parentheses',
]

report = analyzer.analyze(sample_texts)
for category in ['repetition', 'incoherence', 'off_topic', 'grammatical']:
    print(f"{category}: {report[category]['count']} / {report['total_texts']}")
print(f"\nRepetition examples: {report['repetition']['examples']}")